In [10]:
from langchain_huggingface import HuggingFaceEndpoint, HuggingFaceEndpointEmbeddings, HuggingFaceEmbeddings, ChatHuggingFace
from langgraph.graph import StateGraph ,START, END
from typing import TypedDict, Literal
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import os

In [11]:
load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")
HUGGINGFACE_REPO_ID = "Qwen/Qwen2.5-7B-Instruct"

if not HF_TOKEN:
    raise ValueError("HF_TOKEN is missing. Add it to your .env file or environment variables.")

In [12]:
class sentiment_schema(BaseModel):
    sentiment: Literal["positive", "negative"] = Field(
        description="The sentiment of the review; choose exactly 'positive' or 'negative'."
    )


llm = HuggingFaceEndpoint(
    repo_id=HUGGINGFACE_REPO_ID,
    task="text-generation",
    provider="featherless-ai",
    huggingfacehub_api_token=HF_TOKEN,
    temperature=0.0,
)

model = ChatHuggingFace(llm=llm, verbose=True)

structured_model = model.with_structured_output(
    sentiment_schema,
    method="json_schema",
)


In [13]:
def sentiment_prompt_template(review: str) -> str:
    return f"""You are a strict sentiment classifier.
Return only one label: positive or negative.

Classification rule:
- positive: review mainly praises the product or experience
- negative: review mainly criticizes the product or experience

Review:
{review}
"""


def get_sentiment_from_review(review: str):
    response = structured_model.invoke(sentiment_prompt_template(review))

    if isinstance(response, dict):
        sentiment = response.get("sentiment")
    else:

        sentiment = getattr(response, "sentiment", None)    
    return str(sentiment).lower()





In [14]:
def sentiment_prompt_template(review: str) -> str:
    return f"""Classify the sentiment of the following review. Return only one of these values: positive or negative.

Review:
{review}
"""


def get_sentiment_from_review(review: str):
    response = structured_model.invoke(sentiment_prompt_template(review))
    # if isinstance(response, dict):
    #     sentiment = response.get("sentiment")
    # else:
    #     sentiment = getattr(response, "sentiment", None)

    # if sentiment is None:
    #     raise ValueError(f"Unexpected response format: {response!r}")

    return response.get("sentiment").lower()


reviews = [
    "I absolutely loved this product! The quality is excellent, the design is beautiful, and it works perfectly. The delivery was fast, and the customer service was also very helpful. I highly recommend it to everyone!",
    "I’m very disappointed with this product. The quality is poor, the design feels cheap, and it doesn’t work properly. The delivery was extremely late, and the customer service was unhelpful. I would not recommend this product to anyone.",
]

for i, review in enumerate(reviews, start=1):
    sentiment = get_sentiment_from_review(review)
    print(f"Review {i} sentiment: {sentiment}")

HfHubHTTPError: Client error '402 Payment Required' for url 'https://router.huggingface.co/featherless-ai/v1/chat/completions' (Request ID: Root=1-6ae49f07-1c0dc3fd7a36c15b2330c8c4;109cd19e-210e-4bdd-9daa-86ae54bcff98)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have no remaining credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get monthly included credits.

In [ ]:
class ReviewSentiment(TypedDict):
    review: str
    sentiment: Literal["positive"]
    diagonosis: str
    response:str

In [ ]:
def findsentiment(state: ReviewSentiment):
    prompt = f'for the following review, find out the sentiment {state["review"]}'
    sentiment= get_sentiment_from_review(review)
    return {'sentiment': sentiment}

In [ ]:
graph=StateGraph(ReviewSentiment)
graph.add_node('findsentiment', findsentiment)

graph.add_edge(START, 'findsentiment')
graph.add_edge('findsentiment', END)

workflow=graph.compile()

In [ ]:
from IPython.display import Image
Image(workflow.get_graph().draw_mermaid_png())

In [ ]:
input_review ={
    "review": """I absolutely loved this product! The quality is excellent, the design is beautiful, and it works perfectly. 
    The delivery was fast, and the customer service was also very helpful. I highly recommend it to everyone!"""}


workflow.invoke(input_review)
